<a href="https://colab.research.google.com/github/talahazo/EM.Python/blob/Intento-1/Proyecto_MAT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Proyecto Parte 1


In [ ]:
%%writefile mpea_model.py
from __future__ import annotations

import argparse
import json
import re
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

TARGET_HINT = 'PROPERTY: YS (MPa)'
PROCESS_FEATURE = 'ProcessingMethod'
DENSITY_FEATURE = 'Density_g_cm3'
TEMP_FEATURE = 'TestTemperature_C'
ELEMENT_PREFIX = 'X_'

# Peso atómico (g/mol) y densidad de los elementos más habituales en MPEAs.
# Para elementos fuera de esta tabla se conserva la composición; la densidad se imputa.
ELEMENT_DATA = {
    'Al': (26.9815, 2.70), 'Co': (58.9332, 8.90), 'Cr': (51.9961, 7.19),
    'Cu': (63.546, 8.96), 'Fe': (55.845, 7.87), 'Hf': (178.49, 13.31),
    'Mn': (54.938, 7.21), 'Mo': (95.95, 10.28), 'Nb': (92.906, 8.57),
    'Ni': (58.6934, 8.91), 'Re': (186.207, 21.02), 'Si': (28.085, 2.33),
    'Sn': (118.710, 7.31), 'Ta': (180.948, 16.69), 'Ti': (47.867, 4.51),
    'V': (50.9415, 6.11), 'W': (183.84, 19.25), 'Zr': (91.224, 6.52),
    'Mg': (24.305, 1.74), 'Zn': (65.38, 7.14), 'Ag': (107.868, 10.49),
    'Au': (196.967, 19.32), 'B': (10.81, 2.34), 'C': (12.011, 2.27),
    'Ce': (140.116, 6.77), 'La': (138.905, 6.15), 'Sc': (44.956, 2.99),
    'Y': (88.906, 4.47), 'Pd': (106.42, 12.02), 'Pt': (195.084, 21.45),
    'Ga': (69.723, 5.91), 'Ge': (72.630, 5.32), 'In': (114.818, 7.31),
    'Ir': (192.217, 22.56), 'Ru': (101.07, 12.37), 'Rh': (102.906, 12.41),
    'Os': (190.23, 22.59), 'Pb': (207.2, 11.34), 'P': (30.974, 1.82),
    'S': (32.06, 2.07), 'As': (74.922, 5.73), 'Bi': (208.980, 9.78),
    'Li': (6.94, 0.53), 'Be': (9.012, 1.85), 'Cd': (112.414, 8.65),
    'Ir': (192.217, 22.56), 'Th': (232.038, 11.72), 'U': (238.029, 19.05),
}
ELEMENTS = set('H He Li Be B C N O F Ne Na Mg Al Si P S Cl Ar K Ca Sc Ti V Cr Mn Fe Co Ni Cu Zn Ga Ge As Se Br Kr Rb Sr Y Zr Nb Mo Tc Ru Rh Pd Ag Cd In Sn Sb Te I Xe Cs Ba La Ce Pr Nd Pm Sm Eu Gd Tb Dy Ho Er Tm Yb Lu Hf Ta W Re Os Ir Pt Au Hg Tl Pb Bi Po At Rn Fr Ra Ac Th Pa U Np Pu Am Cm Bk Cf Es Fm Md No Lr Rf Db Sg Bh Hs Mt Ds Rg Cn Nh Fl Mc Lv Ts Og'.split())
FORMULA_RE = re.compile(r'([A-Z][a-z]?)(?:\s*((?:\d+(?:\.\d*)?|\.\d+)))?')


def parse_formula(formula):
    """Convierte Al0.25Co1Fe1Ni1 en cantidades por elemento."""
    if formula is None or (isinstance(formula, float) and np.isnan(formula)):
        return {}
    text = str(formula).translate(str.maketrans('₀₁₂₃₄₅₆₇₈₉', '0123456789')).strip()
    amounts = {}
    matches = list(FORMULA_RE.finditer(text))
    if not matches:
        return {}
    # No aceptar texto arbitrario entre símbolos; se permiten espacios y separadores.
    end = 0
    for match in matches:
        gap = text[end:match.start()]
        if gap.strip(' ,;·|+-'):
            return {}
        symbol, raw_amount = match.groups()
        if symbol not in ELEMENTS:
            return {}
        amount = float(raw_amount) if raw_amount else 1.0
        if amount <= 0:
            return {}
        amounts[symbol] = amounts.get(symbol, 0.0) + amount
        end = match.end()
    if text[end:].strip(' ,;·|+-'):
        return {}
    total = sum(amounts.values())
    return {element: value / total for element, value in amounts.items()} if total else {}


def calculate_density(composition):
    """Estimación ideal por regla de mezclas a partir de fracciones atómicas."""
    comp = {e: float(x) for e, x in composition.items() if x > 0}
    if not comp or any(e not in ELEMENT_DATA for e in comp):
        return np.nan
    total = sum(comp.values())
    comp = {e: x / total for e, x in comp.items()}
    mean_mass = sum(x * ELEMENT_DATA[e][0] for e, x in comp.items())
    molar_volume = sum(x * ELEMENT_DATA[e][0] / ELEMENT_DATA[e][1] for e, x in comp.items())
    return mean_mass / molar_volume if molar_volume else np.nan


def canonicalize_processing(value):
    text = str(value or 'UNKNOWN').strip().upper()
    compact = re.sub(r'[^A-Z]', '', text)
    if any(k in compact for k in ('POWDER', 'SPS', 'SPARKPLASMA', 'PM', 'MECHANICALALLOY')):
        return 'POWDER'
    if any(k in compact for k in ('WROUGHT', 'FORGING', 'FORGED', 'ROLLING', 'ROLLED', 'EXTRUSION', 'EXTRUDED')):
        return 'WROUGHT'
    if any(k in compact for k in ('ANNEAL', 'HOMOGEN', 'HEATTREAT', 'SOLUTIONTREAT')):
        return 'ANNEAL'
    if any(k in compact for k in ('CAST', 'CASTING', 'ARC MELT', 'ARCMELT', 'MELTING', 'INGOT')):
        return 'CAST'
    return text or 'UNKNOWN'


def _norm(value):
    return re.sub(r'[^a-z0-9]+', '', str(value).lower())


def find_target_column(columns):
    exact = [_norm(c) for c in columns]
    for preferred in ('propertyysmpa', 'ysmpa', 'yieldstrengthmpa'):
        for col, normalized in zip(columns, exact):
            if normalized == preferred:
                return col
    candidates = [c for c, n in zip(columns, exact) if ('yield' in n or 'ys' in n) and ('mpa' in n or 'strength' in n)]
    if candidates:
        return candidates[0]
    raise ValueError(f'No encontré la columna objetivo {TARGET_HINT!r}. Columnas: {list(columns)}')


def find_composition_column(frame):
    columns = list(frame.columns)
    for col in columns:
        n = _norm(col)
        if any(word in n for word in ('composition', 'formula', 'stoichiometry', 'chemicalformula', 'alloy')):
            return col
    # Fallback: elegir la columna cuyos valores parecen fórmulas químicas.
    best_col, best_score = None, 0.0
    for col in columns:
        sample = frame[col].dropna().astype(str).head(100)
        if len(sample) == 0:
            continue
        score = sum(bool(parse_formula(v)) for v in sample) / len(sample)
        if score > best_score:
            best_col, best_score = col, score
    if best_col is None or best_score < 0.25:
        raise ValueError('No detecté la columna de fórmula/composición química. Define COMPOSITION_COLUMN manualmente en el código.')
    return best_col


def find_optional_column(columns, keywords, exclude=()):
    scored = []
    for col in columns:
        n = _norm(col)
        if any(x in n for x in exclude):
            continue
        score = sum(2 if n == k else 1 for k in keywords if k in n)
        if score:
            scored.append((score, col))
    return sorted(scored, reverse=True, key=lambda item: item[0])[0][1] if scored else None


def make_feature_frame(compositions, elements, temperatures, processing, density_fallback=0.0):
    rows = []
    for comp, temp, method in zip(compositions, temperatures, processing):
        normalized = {e: float(v) for e, v in comp.items() if v > 0}
        total = sum(normalized.values())
        normalized = {e: v / total for e, v in normalized.items()} if total else {}
        density = calculate_density(normalized)
        if not np.isfinite(density):
            density = float(density_fallback)
        row = {f'{ELEMENT_PREFIX}{e}': normalized.get(e, 0.0) for e in elements}
        row[DENSITY_FEATURE] = density
        row[TEMP_FEATURE] = float(temp) if temp is not None and np.isfinite(float(temp)) else 25.0
        row[PROCESS_FEATURE] = canonicalize_processing(method)
        rows.append(row)
    return pd.DataFrame(rows)


def train(data_path, output_path='artifacts/mpea_model.joblib', random_state=42):
    frame = pd.read_csv(data_path, encoding='utf-8-sig')
    target_col = find_target_column(frame.columns)
    composition_col = find_composition_column(frame)
    temperature_col = find_optional_column(frame.columns, ('testtemperature', 'testingtemperature', 'temperature', 'temp'), exclude=('melting', 'anneal', 'homogen', 'sinter'))
    processing_col = find_optional_column(frame.columns, ('processingmethod', 'processing', 'process', 'fabrication', 'route', 'method'))

    frame[target_col] = pd.to_numeric(frame[target_col].astype(str).str.replace(',', '', regex=False), errors='coerce')
    frame = frame.dropna(subset=[target_col]).copy()
    compositions = frame[composition_col].map(parse_formula)
    valid = compositions.map(bool)
    frame, compositions = frame.loc[valid].copy(), compositions.loc[valid]
    if len(frame) < 5:
        raise ValueError(f'Solo quedaron {len(frame)} filas utilizables; se requieren al menos 5 con YS y fórmula válida.')

    elements = sorted({element for comp in compositions for element in comp})
    if not elements:
        raise ValueError('No pude extraer elementos de la columna de composición.')
    temperatures = pd.to_numeric(frame[temperature_col], errors='coerce') if temperature_col else pd.Series(25.0, index=frame.index)
    if temperatures.notna().any():
        temperature_default = float(temperatures.median())
    else:
        temperature_default = 25.0
    temperatures = temperatures.fillna(temperature_default)
    methods = frame[processing_col].map(canonicalize_processing) if processing_col else pd.Series('UNKNOWN', index=frame.index)
    densities = pd.Series([calculate_density(comp) for comp in compositions], index=frame.index, dtype=float)
    density_default = float(densities.median()) if densities.notna().any() else 0.0
    X = make_feature_frame(compositions, elements, temperatures, methods, density_default)
    y = frame[target_col].astype(float).reset_index(drop=True)
    X = X.reset_index(drop=True)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=random_state)

    element_cols = [f'{ELEMENT_PREFIX}{e}' for e in elements]
    numeric_cols = [DENSITY_FEATURE, TEMP_FEATURE]
    preprocess = ColumnTransformer([
        ('composition', SimpleImputer(strategy='median'), element_cols),
        ('numeric', SimpleImputer(strategy='median'), numeric_cols),
        ('processing', Pipeline([
            ('imputer', SimpleImputer(strategy='constant', fill_value='UNKNOWN')),
            ('onehot', OneHotEncoder(handle_unknown='ignore')),
        ]), [PROCESS_FEATURE]),
    ])
    pipeline = Pipeline([
        ('preprocess', preprocess),
        ('model', RandomForestRegressor(n_estimators=500, min_samples_leaf=2, random_state=random_state, n_jobs=-1)),
    ])

    cv_results = {}
    folds = min(5, len(X_train))
    if folds >= 2:
        cv = KFold(n_splits=folds, shuffle=True, random_state=random_state)
        scores = cross_validate(pipeline, X_train, y_train, cv=cv,
                                scoring={'r2': 'r2', 'rmse': 'neg_root_mean_squared_error'},
                                n_jobs=-1, error_score='raise')
        cv_results = {
            'cv_r2_mean': float(np.nanmean(scores['test_r2'])),
            'cv_r2_std': float(np.nanstd(scores['test_r2'])),
            'cv_rmse_mean': float(-np.mean(scores['test_rmse'])),
            'cv_folds': folds,
        }

    pipeline.fit(X_train, y_train)
    prediction = pipeline.predict(X_test)
    metrics = {
        **cv_results,
        'test_r2': float(r2_score(y_test, prediction)),
        'test_rmse': float(np.sqrt(mean_squared_error(y_test, prediction))),
        'n_rows': int(len(frame)),
        'n_elements': int(len(elements)),
    }
    feature_names = pipeline.named_steps['preprocess'].get_feature_names_out().tolist()
    importances = pipeline.named_steps['model'].feature_importances_.tolist()
    output = Path(output_path)
    output.parent.mkdir(parents=True, exist_ok=True)
    joblib.dump({
        'pipeline': pipeline,
        'elements': elements,
        'metrics': metrics,
        'temperature_default': temperature_default,
        'density_default': density_default,
        'processing_categories': sorted(set(methods)),
        'feature_names': feature_names,
        'feature_importances': importances,
        'y_test': np.asarray(y_test),
        'y_pred': np.asarray(prediction),
        'composition_column': str(composition_col),
        'target_column': str(target_col),
    }, output)
    print(f'Dataset: {data_path} | filas usadas: {len(frame)} | composición: {composition_col} | objetivo: {target_col}')
    print('Métricas:', json.dumps(metrics, indent=2, ensure_ascii=False))
    print(f'Modelo guardado en: {output.resolve()}')
    return output


if __name__ == '__main__':
    parser = argparse.ArgumentParser(description='Entrena un predictor de límite elástico para MPEAs.')
    parser.add_argument('--data', default='MPEA_dataset - MPEA_dataset.csv', help='Ruta al CSV de MPEAs')
    parser.add_argument('--output', default='artifacts/mpea_model.joblib', help='Archivo del modelo serializado')
    parser.add_argument('--seed', type=int, default=42)
    args = parser.parse_args()
    train(args.data, args.output, args.seed)


Writing mpea_model.py


In [ ]:
%%writefile app.py
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import streamlit as st

from mpea_model import (
    PROCESS_FEATURE, calculate_density, canonicalize_processing,
    make_feature_frame, parse_formula,
)

MODEL_PATH = Path('artifacts/mpea_model.joblib')
st.set_page_config(page_title='MPEA YS Predictor', page_icon='⚙️', layout='wide')
st.markdown('''
<style>
.block-container {max-width: 1180px; padding-top: 1.6rem;}
[data-testid="stMetric"] {background:#f4f7fb; border:1px solid #e4eaf2; padding:16px; border-radius:14px; color:#000000 !important;}
[data-testid="stMetric"] * {color:#000000 !important; -webkit-text-fill-color:#000000 !important; opacity:1 !important;}
.small-note {color:#667085; font-size:0.92rem;}
</style>
''', unsafe_allow_html=True)

st.title('Predictor de límite elástico para MPEAs')
st.caption('Estimación de Yield Strength (YS) a partir de composición, proceso y temperatura de ensayo.')

if not MODEL_PATH.exists():
    st.warning('Todavía no existe el modelo entrenado. Ejecuta primero las celdas de instalación y entrenamiento del notebook.')
    st.stop()

bundle = joblib.load(MODEL_PATH)
pipeline = bundle['pipeline']
elements = bundle['elements']
metrics = bundle['metrics']
known_methods = {'CAST', 'ANNEAL', 'POWDER', 'WROUGHT'} | set(bundle.get('processing_categories', []))

with st.sidebar:
    st.header('Parámetros')
    input_mode = st.radio('Composición', ['Porcentajes atómicos', 'Fórmula química'], horizontal=False)
    st.markdown('<p class="small-note">Los porcentajes se normalizan automáticamente antes de predecir.</p>', unsafe_allow_html=True)

st.subheader('Composición y procesamiento')
with st.form('mpea_input'):
    if input_mode == 'Porcentajes atómicos':
        st.write('Ajusta los elementos detectados en el dataset (at.%).')
        values = {}
        cols = st.columns(3)
        for index, element in enumerate(elements):
            with cols[index % 3]:
                values[element] = st.slider(f'{element} (at.%)', min_value=0, max_value=100, value=0, step=1, key=f'fraction_{element}')
    else:
        formula_text = st.text_input('Fórmula estequiométrica', value='Al0.25Co1Fe1Ni1', help='Ejemplo: Al0.25Co1Fe1Ni1')
    left, right = st.columns(2)
    with left:
        method = st.selectbox('Método de procesamiento', sorted(known_methods), index=sorted(known_methods).index('CAST') if 'CAST' in known_methods else 0)
    with right:
        temp_default = float(bundle.get('temperature_default', 25.0))
        temperature = st.number_input('Temperatura de ensayo (°C)', min_value=-273.0, max_value=2000.0, value=temp_default, step=10.0)
    submitted = st.form_submit_button('Calcular Propiedades Mecánicas', type='primary', use_container_width=True)

if submitted:
    if input_mode == 'Porcentajes atómicos':
        composition = {element: float(value) for element, value in values.items() if value > 0}
    else:
        composition = parse_formula(formula_text)
    if not composition:
        st.error('No pude leer una composición válida. Usa símbolos químicos válidos y cantidades positivas.')
    else:
        unknown = sorted(set(composition) - set(elements))
        if unknown:
            st.warning('El modelo no vio estos elementos durante el entrenamiento y no puede aprender sus efectos: ' + ', '.join(unknown) + '. Se ignorarán.')
        composition = {e: x for e, x in composition.items() if e in elements}
        if not composition:
            st.error('La composición solo contiene elementos ausentes del dataset de entrenamiento.')
        else:
            total = sum(composition.values())
            composition = {e: x / total for e, x in composition.items()}
            features = make_feature_frame([composition], elements, [temperature], [method], bundle.get('density_default', 0.0))
            prediction = float(pipeline.predict(features)[0])
            st.session_state['mpea_prediction'] = prediction
            st.session_state['mpea_composition'] = composition
            st.session_state['mpea_density'] = calculate_density(composition)

prediction = st.session_state.get('mpea_prediction')
if prediction is not None:
    composition = st.session_state.get('mpea_composition', {})
    col1, col2, col3 = st.columns([1.3, 1, 1])
    with col1:
        st.metric('YS predicho', f'{prediction:,.1f} MPa')
    with col2:
        density = st.session_state.get('mpea_density')
        st.metric('Densidad estimada', f'{density:.2f} g/cm³' if density is not None and np.isfinite(density) else 'No disponible')
    with col3:
        st.metric('R² en prueba', f"{metrics['test_r2']:.3f}")
    formula_display = ' '.join(f'{e}{x:.3f}' for e, x in sorted(composition.items()))
    st.success(f'Composición normalizada: {formula_display}')
else:
    st.info('Introduce una composición y pulsa «Calcular Propiedades Mecánicas» para obtener la estimación.')

st.divider()
tab_prediction, tab_model = st.tabs(['Predicción', 'Comportamiento del modelo'])
with tab_prediction:
    st.markdown('### Resumen del modelo')
    m1, m2, m3 = st.columns(3)
    m1.metric('Filas utilizadas', f"{metrics['n_rows']:,}")
    m2.metric('RMSE en prueba', f"{metrics['test_rmse']:.1f} MPa")
    m3.metric('Elementos aprendidos', str(metrics['n_elements']))
    st.caption('La predicción es una estimación dentro del dominio del dataset. La densidad se aproxima por regla de mezclas y se imputa cuando faltan propiedades elementales.')

with tab_model:
    st.markdown('### Valores reales vs. predichos')
    y_test = np.asarray(bundle['y_test'])
    y_pred = np.asarray(bundle['y_pred'])
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(y_test, y_pred, alpha=0.72, color='#2878B5', edgecolor='white', linewidth=0.5)
    low = float(min(np.min(y_test), np.min(y_pred)))
    high = float(max(np.max(y_test), np.max(y_pred)))
    ax.plot([low, high], [low, high], linestyle='--', color='#D1495B', label='Ajuste perfecto (y = x)')
    ax.set(xlabel='YS real (MPa)', ylabel='YS predicho (MPa)', title='Conjunto de prueba')
    ax.legend(frameon=False)
    ax.grid(alpha=0.2)
    st.pyplot(fig, use_container_width=True)
    plt.close(fig)

    st.markdown('### Importancia de características')
    names = bundle['feature_names']
    importances = np.asarray(bundle['feature_importances'])
    labels = []
    for name in names:
        label = name.replace('composition__X_', 'x_').replace('numeric__', '').replace('processing__ProcessingMethod_', 'Proceso: ')
        labels.append(label)
    order = np.argsort(importances)[-15:]
    fig, ax = plt.subplots(figsize=(8, max(4, 0.32 * len(order))))
    ax.barh([labels[i] for i in order], importances[order], color='#4C9F70')
    ax.set_xlabel('Importancia del Random Forest')
    ax.set_title('15 características más influyentes')
    ax.grid(axis='x', alpha=0.2)
    st.pyplot(fig, use_container_width=True)
    plt.close(fig)

    if 'cv_rmse_mean' in metrics:
        st.caption(f"Validación cruzada ({metrics['cv_folds']} folds): RMSE medio {metrics['cv_rmse_mean']:.1f} MPa; R² medio {metrics['cv_r2_mean']:.3f}.")


Overwriting app.py


In [ ]:
# Ejecuta primero las dos celdas %%writefile para crear mpea_model.py y app.py.
# Instala dependencias, usa Dataset/ y entrena con las columnas reales del CSV.
%pip install -q pandas scikit-learn streamlit matplotlib seaborn joblib

from pathlib import Path

DATASET_FILENAME = 'MPEA_dataset - MPEA_dataset.csv'
candidates = [
    Path('Dataset') / DATASET_FILENAME,
    Path('/content/Dataset') / DATASET_FILENAME,
    Path('/content/drive/MyDrive/Dataset') / DATASET_FILENAME,
    Path(DATASET_FILENAME),
]
DATASET_PATH = next((path for path in candidates if path.exists()), None)
if DATASET_PATH is None:
    from google.colab import files
    print(f'No encontré Dataset/{DATASET_FILENAME}. Selecciona el CSV para cargarlo.')
    uploaded = files.upload()
    csv_files = [Path(name) for name in uploaded if name.lower().endswith('.csv')]
    if not csv_files:
        raise FileNotFoundError('No se cargó ningún archivo CSV.')
    DATASET_PATH = next((path for path in csv_files if path.name == DATASET_FILENAME), csv_files[0])

# Este dataset incluye PROPERTY: Calculated Density. Si está disponible, se usa;
# las filas faltantes recurren a la estimación por regla de mezclas del modelo.
model_file = Path('mpea_model.py')
source = model_file.read_text(encoding='utf-8')
old_density_block = (
    "    densities = pd.Series([calculate_density(comp) for comp in compositions], index=frame.index, dtype=float)\n"
    "    density_default = float(densities.median()) if densities.notna().any() else 0.0\n"
    "    X = make_feature_frame(compositions, elements, temperatures, methods, density_default)"
)
new_density_block = (
    "    density_col = find_optional_column(frame.columns, ('calculateddensity', 'density'), exclude=('experimental', 'expdensity'))\n"
    "    ideal_densities = pd.Series([calculate_density(comp) for comp in compositions], index=frame.index, dtype=float)\n"
    "    if density_col:\n"
    "        reported = pd.to_numeric(frame[density_col], errors='coerce')\n"
    "        densities = reported.where(reported.notna(), ideal_densities)\n"
    "    else:\n"
    "        densities = ideal_densities\n"
    "    density_default = float(densities.median()) if densities.notna().any() else 0.0\n"
    "    X = make_feature_frame(compositions, elements, temperatures, methods, density_default)\n"
    "    X[DENSITY_FEATURE] = densities.fillna(density_default).to_numpy()"
)
if old_density_block not in source:
    raise RuntimeError('No encontré el bloque de densidad esperado en mpea_model.py; revisa que la primera celda se haya ejecutado.')
model_file.write_text(source.replace(old_density_block, new_density_block, 1), encoding='utf-8')

import sys
sys.modules.pop('mpea_model', None)
from mpea_model import train
print(f'CSV seleccionado: {DATASET_PATH}')
MODEL_PATH = train(DATASET_PATH, output_path='artifacts/mpea_model.joblib', random_state=42)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 37.2 MB/s eta 0:00:00
No encontré Dataset/MPEA_dataset - MPEA_dataset.csv. Selecciona el CSV para cargarlo.


Saving MPEA_dataset - MPEA_dataset.csv to MPEA_dataset - MPEA_dataset.csv
CSV seleccionado: MPEA_dataset - MPEA_dataset.csv
Dataset: MPEA_dataset - MPEA_dataset.csv | filas usadas: 1067 | composición: FORMULA | objetivo: PROPERTY: YS (MPa)
Métricas: {
  "cv_r2_mean": 0.751211196322952,
  "cv_r2_std": 0.029214149859159363,
  "cv_rmse_mean": 280.0172834526214,
  "cv_folds": 5,
  "test_r2": 0.6626110713339641,
  "test_rmse": 340.9528326913736,
  "n_rows": 1067,
  "n_elements": 26
}
Modelo guardado en: /content/artifacts/mpea_model.joblib


In [ ]:
# Reinicia Streamlit con la configuración compatible con el proxy de Colab.
import socket
import subprocess
import time
from google.colab import output

try:
    if streamlit_process.poll() is None:
        streamlit_process.terminate()
        streamlit_process.wait(timeout=10)
except NameError:
    pass
except subprocess.TimeoutExpired:
    streamlit_process.kill()
    streamlit_process.wait(timeout=5)

try:
    streamlit_log.close()
except NameError:
    pass

streamlit_log = open('/content/streamlit.log', 'w')
streamlit_process = subprocess.Popen(
    [
        'streamlit', 'run', 'app.py',
        '--server.port=8501',
        '--server.address=0.0.0.0',
        '--server.headless=true',
        '--server.enableCORS=false',
        '--server.enableXsrfProtection=false',
    ],
    stdout=streamlit_log,
    stderr=subprocess.STDOUT,
)

for _ in range(30):
    if streamlit_process.poll() is not None:
        raise RuntimeError('Streamlit terminó al iniciar. Revisa /content/streamlit.log')
    try:
        with socket.create_connection(('127.0.0.1', 8501), timeout=1):
            break
    except OSError:
        time.sleep(1)
else:
    raise TimeoutError('Streamlit no abrió el puerto 8501. Revisa /content/streamlit.log')

time.sleep(2)
print('Servidor Streamlit listo en puerto 8501; proxy Colab compatible habilitado.')
output.serve_kernel_port_as_iframe(8501, height=850)

Servidor Streamlit listo en puerto 8501; proxy Colab compatible habilitado.


<IPython.core.display.Javascript object>

## MPEA YS Predictor

Aplicación interactiva para estimar el límite elástico (**YS, MPa**) de aleaciones MPEA a partir de composición, procesamiento y temperatura de ensayo.

### Cómo ejecutarlo
1. Ejecuta primero las dos celdas `%%writefile` para crear `mpea_model.py` y `app.py`.
2. Ejecuta la celda de instalación y entrenamiento. Busca `Dataset/MPEA_dataset - MPEA_dataset.csv` (también revisa rutas habituales de Drive); si no lo encuentra, te pedirá cargar el CSV.
3. Ejecuta la última celda de código para abrir Streamlit dentro de Colab.

**Columnas del CSV detectadas:** `FORMULA`, `PROPERTY: YS (MPa)`, `PROPERTY: Processing method`, `PROPERTY: Test temperature ($^\circ$C)` y `PROPERTY: Calculated Density (g/cm$^3$)`. El entrenamiento usa Random Forest, separación 80/20, validación cruzada, R² y RMSE; guarda el pipeline en `artifacts/mpea_model.joblib`.

**Dependencias:** `pandas`, `scikit-learn`, `streamlit`, `matplotlib`, `seaborn` y `joblib`. Usa la densidad calculada del CSV cuando está disponible; para valores faltantes estima por regla de mezclas. En la app, la densidad se estima desde la composición. Los controles de composición muestran los elementos presentes en el dataset de entrenamiento.


# Comentarios


In [ ]:
#Aquí puedes poner tu comentarios con el gato # y el texto xd